# 🏆 NLBSE'26 - FIXED VERSION (Simpler, Actually Works)

## What was wrong with the previous version:
1. ❌ Encoder layers not unfreezing properly for ModernBERT
2. ❌ LSAN too complex, hurting performance
3. ❌ Learning rate too low
4. ❌ Ensemble too heavy (281 GFLOPS vs 100 max)

## This version:
1. ✅ Simple, proven architecture
2. ✅ Proper fine-tuning (unfreeze more layers)
3. ✅ Asymmetric Loss for class imbalance
4. ✅ Per-label threshold optimization
5. ✅ Lighter ensemble (2-3 models max)

In [ ]:
# ============================================
# CELL 1: Verify Setup
# ============================================
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ============================================
# CELL 2: Imports
# ============================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import json
import gc
import time
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from datasets import load_dataset
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler

# Seed
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# ============================================
# CELL 3: Configuration
# ============================================

# UPDATE THESE PATHS
OUTPUT_DIR = r"D:\NLBSE code comment classification\nlbse26_fixed_output"
PREPROCESSED_DIR = "/content/drive/MyDrive/preprocessed"  # or your local path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# USE ONLY 2 MODELS FOR EFFICIENCY (under 100 GFLOPS)
MODEL_CONFIGS = {
    "codebert": {
        "name": "microsoft/codebert-base",
        "max_length": 256,
        "batch_size": 16,
        "unfreeze_layers": 4,  # Unfreeze top 4 layers (was 2 - too few!)
    },
    "graphcodebert": {
        "name": "microsoft/graphcodebert-base",
        "max_length": 256,
        "batch_size": 16,
        "unfreeze_layers": 4,
    },
}

# Labels
LANGUAGES = ['java', 'python', 'pharo']
LABEL_NAMES = {
    'java': ['summary', 'Ownership', 'Expand', 'usage', 'Pointer', 'deprecation', 'rational'],
    'python': ['Usage', 'Parameters', 'DevelopmentNotes', 'Expand', 'Summary'],
    'pharo': ['Keyimplementationpoints', 'Example', 'Responsibilities', 'Intent', 'Keymessages', 'Collaborators']
}

ALL_LABELS = []
for lang in LANGUAGES:
    ALL_LABELS.extend([f"{lang}_{l}" for l in LABEL_NAMES[lang]])
NUM_LABELS = len(ALL_LABELS)

# Training - INCREASED EPOCHS AND LR
EPOCHS = 10  # More epochs!
LR = 3e-5    # Higher LR
PATIENCE = 4

print(f"Labels: {NUM_LABELS}")
print(f"Models: {list(MODEL_CONFIGS.keys())}")

In [ ]:
# ============================================
# CELL 4: Load Data
# ============================================

def load_data():
    """Load from HuggingFace (most reliable)."""
    print("Loading from HuggingFace...")
    ds = load_dataset("NLBSE/nlbse25-code-comment-classification")
    
    train_dfs, test_dfs = [], []
    for lang in LANGUAGES:
        if f"{lang}_train" in ds:
            tr = ds[f"{lang}_train"].to_pandas()
            te = ds[f"{lang}_test"].to_pandas()
            tr['language'] = lang
            te['language'] = lang
            tr['text'] = tr['class'].fillna('') + " | " + tr['comment_sentence'].fillna('')
            te['text'] = te['class'].fillna('') + " | " + te['comment_sentence'].fillna('')
            train_dfs.append(tr)
            test_dfs.append(te)
            print(f"  {lang}: {len(tr)} train, {len(te)} test")
    
    train_df = pd.concat(train_dfs, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
    test_df = pd.concat(test_dfs, ignore_index=True)
    return train_df, test_df

def to_unified_labels(row):
    """Convert to 18-label format."""
    lang = row['language']
    labels = np.array(row['labels']) if 'labels' in row and row['labels'] is not None else np.zeros(len(LABEL_NAMES[lang]))
    unified = np.zeros(NUM_LABELS, dtype=np.float32)
    offset = {'java': 0, 'python': 7, 'pharo': 12}[lang]
    for i, v in enumerate(labels):
        if offset + i < NUM_LABELS:
            unified[offset + i] = v
    return unified

train_df, test_df = load_data()
train_df['unified_labels'] = train_df.apply(to_unified_labels, axis=1)
test_df['unified_labels'] = test_df.apply(to_unified_labels, axis=1)

# Label counts for weighted loss
label_counts = np.array([l for l in train_df['unified_labels']]).sum(axis=0)
print(f"\nTotal: {len(train_df)} train, {len(test_df)} test")

In [ ]:
# ============================================
# CELL 5: Asymmetric Loss (PROVEN TO WORK)
# ============================================

class AsymmetricLoss(nn.Module):
    """ASL - handles class imbalance well."""
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip

    def forward(self, x, y):
        x_sigmoid = torch.sigmoid(x)
        xs_pos = x_sigmoid
        xs_neg = 1 - x_sigmoid
        
        if self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1)
        
        loss_pos = y * torch.log(xs_pos.clamp(min=1e-8))
        loss_neg = (1 - y) * torch.log(xs_neg.clamp(min=1e-8))
        loss = loss_pos + loss_neg
        
        pt = xs_pos * y + xs_neg * (1 - y)
        gamma = self.gamma_pos * y + self.gamma_neg * (1 - y)
        loss *= torch.pow(1 - pt, gamma)
        
        return -loss.mean()

print("✅ AsymmetricLoss defined")

In [ ]:
# ============================================
# CELL 6: SIMPLE Classifier (NO LSAN - it was hurting)
# ============================================

class SimpleCodeClassifier(nn.Module):
    """
    Simple but effective classifier.
    Key: Unfreeze MORE layers (4 instead of 2)
    """
    def __init__(self, model_name, num_labels, unfreeze_layers=4, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        
        # Freeze all first
        for param in self.encoder.parameters():
            param.requires_grad = False
        
        # Unfreeze top N layers (THIS IS KEY!)
        if hasattr(self.encoder, 'encoder') and hasattr(self.encoder.encoder, 'layer'):
            layers = self.encoder.encoder.layer
            for layer in layers[-unfreeze_layers:]:
                for param in layer.parameters():
                    param.requires_grad = True
            print(f"  Unfroze top {unfreeze_layers} of {len(layers)} layers")
        
        # Also unfreeze pooler if exists
        if hasattr(self.encoder, 'pooler') and self.encoder.pooler is not None:
            for param in self.encoder.pooler.parameters():
                param.requires_grad = True
        
        # Simple but effective classifier head
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_labels)
        )
        
        # Init
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        
        # Use CLS token
        cls_output = outputs.last_hidden_state[:, 0, :]
        
        return self.classifier(cls_output)

print("✅ SimpleCodeClassifier defined")

In [ ]:
# ============================================
# CELL 7: Dataset
# ============================================

class CommentDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Include language in text
        text = f"[{row['language'].upper()}] {row['text']}"
        
        enc = self.tokenizer(
            text, 
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels': torch.tensor(row['unified_labels'], dtype=torch.float32)
        }

print("✅ Dataset defined")

In [ ]:
# ============================================
# CELL 8: Training Function (FIXED)
# ============================================

def train_model(model_name, config, train_df, val_df, save_path):
    """Train with proper unfreezing and learning rate."""
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")
    
    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(config['name'])
    
    # Model
    model = SimpleCodeClassifier(
        config['name'], 
        NUM_LABELS,
        unfreeze_layers=config['unfreeze_layers']
    ).to(device)
    
    # Count params
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
    
    # Data
    train_ds = CommentDataset(train_df, tokenizer, config['max_length'])
    val_ds = CommentDataset(val_df, tokenizer, config['max_length'])
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=config['batch_size']*2, num_workers=0)
    
    # Loss
    criterion = AsymmetricLoss(gamma_neg=4, gamma_pos=1, clip=0.05)
    
    # Optimizer - DIFFERENT LR for encoder vs head
    encoder_params = [p for n, p in model.named_parameters() if 'encoder' in n and p.requires_grad]
    head_params = [p for n, p in model.named_parameters() if 'encoder' not in n]
    
    optimizer = torch.optim.AdamW([
        {'params': encoder_params, 'lr': LR},      # Encoder LR
        {'params': head_params, 'lr': LR * 5},     # Head gets 5x higher LR
    ], weight_decay=0.01)
    
    # Scheduler
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, 
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )
    
    scaler = GradScaler()
    
    best_f1 = 0
    best_state = None
    patience_counter = 0
    
    for epoch in range(EPOCHS):
        # Train
        model.train()
        total_loss = 0
        
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            
            with autocast():
                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        
        # Validate
        model.eval()
        all_preds, all_labels = [], []
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                
                with autocast():
                    logits = model(input_ids, attention_mask)
                
                preds = (torch.sigmoid(logits) > 0.5).cpu().numpy()
                all_preds.append(preds)
                all_labels.append(batch['labels'].numpy())
        
        all_preds = np.vstack(all_preds)
        all_labels = np.vstack(all_labels)
        
        val_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f}")
        
        # Early stopping
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            print(f"  ✓ New best: {best_f1:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print("  Early stopping!")
                break
    
    # Load best
    if best_state:
        model.load_state_dict(best_state)
    model = model.to(device)
    
    # Save
    Path(save_path).mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), f"{save_path}/model.pt")
    tokenizer.save_pretrained(save_path)
    
    print(f"✅ {model_name} Best F1: {best_f1:.4f}")
    
    gc.collect()
    torch.cuda.empty_cache()
    
    return model, tokenizer, best_f1

print("✅ Training function defined")

In [ ]:
# ============================================
# CELL 9: Threshold Optimization
# ============================================

def optimize_thresholds(probs, labels):
    """Find best threshold per label."""
    print("\nOptimizing thresholds...")
    thresholds = []
    
    for i, name in enumerate(ALL_LABELS):
        best_t, best_f1 = 0.5, 0
        
        # Coarse search
        for t in np.arange(0.1, 0.9, 0.05):
            f1 = f1_score(labels[:, i], (probs[:, i] > t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        
        # Fine search
        for t in np.arange(max(0.05, best_t-0.1), min(0.95, best_t+0.1), 0.01):
            f1 = f1_score(labels[:, i], (probs[:, i] > t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        
        thresholds.append(best_t)
        print(f"  {name:<35} t={best_t:.2f} F1={best_f1:.4f}")
    
    return np.array(thresholds)

print("✅ Threshold optimization defined")

In [ ]:
# ============================================
# CELL 10: Simple Ensemble (2 models only for efficiency)
# ============================================

class SimpleEnsemble:
    """Weighted average of 2 models - efficient!"""
    def __init__(self, models_info):
        self.models = models_info
        
        # Weights based on F1
        total_f1 = sum(m['f1'] for m in models_info.values())
        self.weights = {k: v['f1']/total_f1 for k, v in models_info.items()}
        
        print("Ensemble weights:")
        for k, w in self.weights.items():
            print(f"  {k}: {w:.3f}")
    
    def predict_proba(self, texts, batch_size=32):
        all_probs = []
        
        for key, info in self.models.items():
            model, tokenizer, config = info['model'], info['tokenizer'], info['config']
            model.eval()
            
            probs = []
            for i in range(0, len(texts), batch_size):
                batch = texts[i:i+batch_size]
                enc = tokenizer(batch, padding=True, truncation=True,
                               max_length=config['max_length'], return_tensors='pt')
                enc = {k: v.to(device) for k, v in enc.items()}
                
                with torch.no_grad(), autocast():
                    logits = model(enc['input_ids'], enc['attention_mask'])
                probs.append(torch.sigmoid(logits).cpu().numpy())
            
            all_probs.append(self.weights[key] * np.vstack(probs))
        
        return sum(all_probs)
    
    def predict(self, texts, thresholds, batch_size=32):
        probs = self.predict_proba(texts, batch_size)
        preds = np.array([probs[:, i] > thresholds[i] for i in range(len(thresholds))]).T.astype(int)
        return preds, probs

print("✅ SimpleEnsemble defined")

In [ ]:
# ============================================
# CELL 11: TRAIN MODELS
# ============================================

# Split
train_split, val_split = train_test_split(
    train_df, test_size=0.15, random_state=42, stratify=train_df['language']
)
print(f"Split: {len(train_split)} train, {len(val_split)} val")

# Train both models
trained = {}

for name, config in MODEL_CONFIGS.items():
    model, tokenizer, f1 = train_model(
        name, config, train_split, val_split, f"{OUTPUT_DIR}/{name}"
    )
    trained[name] = {
        'model': model,
        'tokenizer': tokenizer,
        'config': config,
        'f1': f1
    }

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
for k, v in trained.items():
    print(f"  {k}: F1={v['f1']:.4f}")

In [ ]:
# ============================================
# CELL 12: Create Ensemble + Optimize Thresholds
# ============================================

ensemble = SimpleEnsemble(trained)

# Validation predictions
val_texts = [f"[{row['language'].upper()}] {row['text']}" for _, row in val_split.iterrows()]
val_labels = np.array([l for l in val_split['unified_labels']])

print("Getting validation predictions...")
val_probs = ensemble.predict_proba(val_texts, batch_size=32)

# Optimize thresholds
thresholds = optimize_thresholds(val_probs, val_labels)

# Save
thresh_dict = {ALL_LABELS[i]: float(thresholds[i]) for i in range(NUM_LABELS)}
with open(f"{OUTPUT_DIR}/thresholds.json", 'w') as f:
    json.dump(thresh_dict, f, indent=2)
print("\n✅ Thresholds saved")

In [ ]:
# ============================================
# CELL 13: Final Test Evaluation
# ============================================

test_texts = [f"[{row['language'].upper()}] {row['text']}" for _, row in test_df.iterrows()]
test_labels = np.array([l for l in test_df['unified_labels']])

print(f"Evaluating on {len(test_texts)} test samples...")
preds, probs = ensemble.predict(test_texts, thresholds, batch_size=32)

# Metrics
f1_macro = f1_score(test_labels, preds, average='macro', zero_division=0)
f1_micro = f1_score(test_labels, preds, average='micro', zero_division=0)

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"F1 Macro: {f1_macro:.4f}")
print(f"F1 Micro: {f1_micro:.4f}")

# Per category
print("\nPer-Category:")
print("-" * 70)
metrics = []
for i, label in enumerate(ALL_LABELS):
    p = precision_score(test_labels[:, i], preds[:, i], zero_division=0)
    r = recall_score(test_labels[:, i], preds[:, i], zero_division=0)
    f1 = f1_score(test_labels[:, i], preds[:, i], zero_division=0)
    metrics.append({'label': label, 'p': p, 'r': r, 'f1': f1})
    print(f"{label:<35} P={p:.3f} R={r:.3f} F1={f1:.4f}")

print("-" * 70)
print(f"{'AVERAGE':<35} P={np.mean([m['p'] for m in metrics]):.3f} "
      f"R={np.mean([m['r'] for m in metrics]):.3f} F1={np.mean([m['f1'] for m in metrics]):.4f}")

In [ ]:
# ============================================
# CELL 14: Runtime & Submission Score
# ============================================

# Runtime
print("\nMeasuring runtime...")
_ = ensemble.predict_proba(test_texts[:100], 32)  # warmup
torch.cuda.synchronize()

times = []
for _ in range(5):
    torch.cuda.synchronize()
    start = time.time()
    _ = ensemble.predict_proba(test_texts, 32)
    torch.cuda.synchronize()
    times.append((time.time() - start) / len(test_texts))

runtime = np.mean(times)
print(f"Runtime: {runtime*1000:.3f} ms/sample")

# GFLOPS (2 models = ~4 GFLOPS each = ~8 total)
gflops = sum(2 * sum(p.numel() for p in m['model'].parameters()) * 256 / 1e9 
             for m in trained.values())
print(f"GFLOPS: {gflops:.2f}")

# Submission score
f1_comp = 0.60 * f1_macro
rt_comp = 0.20 * max((1.0 - runtime) / 1.0, 0)
gf_comp = 0.20 * max((100.0 - gflops) / 100.0, 0)
total = f1_comp + rt_comp + gf_comp

print("\n" + "="*60)
print("SUBMISSION SCORE")
print("="*60)
print(f"F1 (60%):      {f1_comp:.4f}")
print(f"Runtime (20%): {rt_comp:.4f}")
print(f"GFLOPS (20%):  {gf_comp:.4f}")
print(f"\n🏆 TOTAL: {total:.4f}")

In [ ]:
# ============================================
# CELL 15: Save Results
# ============================================

results = {
    'f1_macro': float(f1_macro),
    'f1_micro': float(f1_micro),
    'runtime_ms': float(runtime * 1000),
    'gflops': float(gflops),
    'submission_score': float(total),
    'models': {k: {'f1': float(v['f1']), 'weight': float(ensemble.weights[k])} 
               for k, v in trained.items()},
    'per_category': metrics,
    'thresholds': thresh_dict
}

with open(f"{OUTPUT_DIR}/results.json", 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Results saved to {OUTPUT_DIR}/results.json")
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"""
Models: CodeBERT + GraphCodeBERT
F1 Macro: {f1_macro:.4f}
Runtime: {runtime*1000:.2f} ms/sample
GFLOPS: {gflops:.2f}

🏆 Submission Score: {total:.4f}
""")